# 真实资产池：连续配权与 MPS-QAOA 混合工作流

本 Notebook 从 `真实资产池.xlsx` 的 610 只资产出发，构造统一的 `portfolio-allocation.v1` 问题契约，并比较四条可复现路径：

1. 全资产池连续松弛 → Top-K → 固定支持集连续再优化；
2. 40 只候选池连续松弛 → Top-K → 固定支持集连续再优化；
3. 40 只候选池联合 MILP（同时选择资产与连续配权）；
4. SA / IBM Aer MPS-QAOA 选择支持集 → 连续配权与完整约束审计。

所有最终权重均为连续变量，不使用固定等权配置。

## 1. 环境与实验参数

默认运行本地 IBM Aer MPS 模拟器。若只想快速检查经典流程，可把 `RUN_MPS` 改为 `False`。

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'lib').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'lib').is_dir():
    raise RuntimeError('未找到同时包含 lib/ 与 problem/ 的仓库根目录。')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from problem.variable_weight_workflow import run_variable_weight_workflow

SOURCE = ROOT / '真实资产池.xlsx'
POLICY = ROOT / '资产配置约束.v1.json'
OUTPUT = ROOT / 'tmp' / 'notebook-real-asset-variable-weight-40'

PROFILE = 'steady'
CANDIDATE_COUNT = 40
HOLDING_COUNT = 10
MINIMUM_ACTIVE_WEIGHT = 0.01
SEED = 7
RUN_MPS = True

assert SOURCE.is_file(), SOURCE
assert POLICY.is_file(), POLICY
print(f'仓库根目录: {ROOT}')
print(f'输出目录: {OUTPUT}')

仓库根目录: C:\Users\admin\OneDrive\Desktop\taiyi\QSolutionData
输出目录: C:\Users\admin\OneDrive\Desktop\taiyi\QSolutionData\tmp\notebook-real-asset-variable-weight-40


## 2. 从完整资产池运行求解流程

40 只候选按五类大类资产均衡配额与综合评分确定，不使用聚类筛选。高相似度的 complete-linkage 分组仅用于生成集中度约束。

In [2]:
report = run_variable_weight_workflow(
    SOURCE,
    OUTPUT,
    profile=PROFILE,
    candidate_count=CANDIDATE_COUNT,
    target_holding_count=HOLDING_COUNT,
    minimum_active_weight=MINIMUM_ACTIVE_WEIGHT,
    policy_path=POLICY,
    seed=SEED,
    run_mps=RUN_MPS,
    solver_time_limit=60.0,
    mps_config={
        'layers': 1,
        'optimizer_iterations': 8,
        'restarts': 1,
        'shots': 4096,
        'max_bond_dimension': 128,
        'truncation_threshold': 1e-8,
        'timeout_seconds': 180.0,
    },
)
print(f"输入资产数: {report['source']['asset_count']}")
print(f"候选资产数: {report['candidate_count']}")
print(f"MPS 已运行: {RUN_MPS}")

输入资产数: 610
候选资产数: 40
MPS 已运行: True


## 3. 方法级结果比较

目标函数越小越好；它是标准化后的选基成本 `-return_score + risk_score - stability_score`，不是直接的收益率。`optimality_scope` 用来区分联合 MILP 全局求解与固定支持集上的连续再优化。

In [3]:
summary_rows = []
for method, result in report['methods'].items():
    proxies = result.get('performance_proxies') or {}
    summary_rows.append({
        'method': method,
        'status': result.get('status'),
        'feasible': result.get('feasible'),
        'holding_count': result.get('holding_count'),
        'objective': result.get('objective'),
        'annualized_return_proxy': proxies.get('annualized_return'),
        'volatility_proxy': proxies.get('volatility'),
        'max_drawdown_proxy': proxies.get('max_drawdown'),
        'runtime_seconds': result.get('runtime_seconds'),
        'optimality_scope': result.get('optimality_scope'),
    })
summary = pd.DataFrame(summary_rows).sort_values(
    ['feasible', 'objective'], ascending=[False, True], na_position='last'
).reset_index(drop=True)
display(summary)
assert summary.loc[summary['status'] != 'not_run', 'feasible'].fillna(False).all()

,method,status,feasible,holding_count,objective,annualized_return_proxy,volatility_proxy,max_drawdown_proxy,runtime_seconds,optimality_scope
0,full_relax_round_reoptimize,optimal,True,10,-1.986338,0.261772,0.019992,0.075885,0.012293,fixed_support
1,candidate_relax_round_reoptimize,optimal,True,10,-1.955206,0.256820,0.020177,0.082440,0.002375,fixed_support
2,candidate_joint_milp,optimal,True,10,-1.955206,0.256820,0.020177,0.082440,0.008736,joint_milp
3,aer_mps_qaoa_selection_then_reoptimize,optimal,True,10,-1.691877,0.237971,0.019258,0.083328,0.001589,fixed_support
4,simulated_annealing_selection_then_reoptimize,optimal,True,10,-1.581635,0.258896,0.022013,0.095418,0.002014,fixed_support


## 4. 查看统一问题契约中的约束

这里直接读取工作流落盘的 40 资产问题契约。权重总和、持仓数、单只上限、资产大类上下限、R5、管理人、二级类型、高相似度组以及绩效代理限制均由求解器和审计器共同使用。

In [4]:
candidate_problem = json.loads(
    (OUTPUT / 'candidate-problem.json').read_text(encoding='utf-8')
)
constraints = candidate_problem['constraints']
constraint_overview = {
    'schema': candidate_problem['schema'],
    'asset_count': len(candidate_problem['assets']),
    'total_weight': constraints['total_weight'],
    'holding_count': constraints['holding_count'],
    'minimum_active_weight': constraints['minimum_active_weight'],
    'single_asset_weight_upper_bound': constraints['single_asset_weight_upper_bound'],
    'r5_weight_upper_bound': constraints['r5_weight_upper_bound'],
    'manager_weight_upper_bound': constraints['manager_weight_upper_bound'],
    'secondary_type_weight_upper_bound': constraints['secondary_type_weight_upper_bound'],
    'high_similarity_group_weight_upper_bound': constraints['high_similarity_group_weight_upper_bound'],
    'high_similarity_group_count': len(constraints['high_similarity_groups']),
    'performance': constraints['performance'],
}
display(pd.Series(constraint_overview, name='value').to_frame())
display(pd.DataFrame(constraints['asset_class_weight_bounds']).T)

,value
schema,portfolio-allocation.v1
asset_count,40
total_weight,1.0
holding_count,"{'min': 8, 'max': 15}"
minimum_active_weight,0.01
single_asset_weight_upper_bound,0.15
r5_weight_upper_bound,0.1
manager_weight_upper_bound,0.25
secondary_type_weight_upper_bound,0.35
high_similarity_group_weight_upper_bound,0.4


,min,max
cash_money,0.05,0.20
fixed_income,0.45,0.75
mixed,0.00,0.25
equity,0.00,0.30
alternative,0.00,0.10


## 5. 查看每条路径的连续权重

工作流为每个成功方法输出独立 CSV。下面验证每条路径的权重和与持仓数，并显示 MPS 路径（若启用）的实际非等权配置。

In [5]:
allocation_files = sorted(OUTPUT.glob('*-allocation.csv'))
allocation_checks = []
allocations = {}
for path in allocation_files:
    method = path.name.removesuffix('-allocation.csv')
    frame = pd.read_csv(path)
    allocations[method] = frame
    allocation_checks.append({
        'method': method,
        'holding_count': len(frame),
        'weight_sum': frame['weight'].sum(),
        'min_weight': frame['weight'].min(),
        'max_weight': frame['weight'].max(),
        'equal_weight': frame['weight'].round(10).nunique() == 1,
    })
checks = pd.DataFrame(allocation_checks)
display(checks)
assert checks['weight_sum'].sub(1.0).abs().lt(1e-7).all()

mps_name = 'aer_mps_qaoa_selection_then_reoptimize'
if mps_name in allocations:
    display(allocations[mps_name].sort_values('weight', ascending=False))
else:
    print('MPS 路径未运行；把 RUN_MPS 设为 True 后重新 Run All。')

,method,holding_count,weight_sum,min_weight,max_weight,equal_weight
0,aer_mps_qaoa_selection_then_reoptimize,10,1.0,0.01,0.15,False
1,candidate_joint_milp,10,1.0,0.05,0.15,False
2,candidate_relax_round_reoptimize,10,1.0,0.05,0.15,False
3,full_relax_round_reoptimize,10,1.0,0.01,0.15,False
4,simulated_annealing_selection_then_reoptimize,10,1.0,0.01,0.15,False


,index,code,security_name,weight,asset_class,investment_type_secondary,risk_level,fund_manager
0,0,007696.OF,嘉实融享浮动净值型,0.15,cash_money,money_market,R1,113
5,13,017498.OF,淳厚添益增强A,0.15,fixed_income,hybrid_bond_secondary,R3,184
8,27,005825.OF,申万菱信智能驱动A,0.15,equity,equity,R4-R5,51
4,12,470010.OF,汇添富多元收益A,0.15,fixed_income,hybrid_bond_secondary,R3,47
7,20,519770.OF,交银优择回报A,0.15,mixed,flexible_allocation,R3-R4,74
2,9,000047.OF,华夏双债增强A,0.14,fixed_income,hybrid_bond_primary,R3,172
6,15,003504.OF,景顺长城景颐丰利A,0.05,fixed_income,hybrid_bond_secondary,R3,135
9,39,002804.OF,华泰柏瑞量化对冲,0.04,alternative,equity_long_short,R3,77
1,8,000122.OF,汇添富实业债A,0.01,fixed_income,hybrid_bond_primary,R3,66
3,10,002405.OF,光大中高等级A,0.01,fixed_income,hybrid_bond_primary,R3,288


## 6. 结论与边界

- `candidate_joint_milp` 是 40 资产范围内同时选择与配权的混合整数线性基线；求解成功且 gap 为零时，可视为该模型精度下的全局最优。
- `full_relax_round_reoptimize` 与 `candidate_relax_round_reoptimize` 的最终解，只保证固定支持集上的连续配权最优。
- MPS-QAOA 求解的是紧凑的支持集选择 QUBO；之后使用相同 `portfolio-allocation.v1` 契约做连续再优化与约束审计。因此它是量子启发式选股 + 经典配权的混合流程，不宣称联合全局最优。
- 当前波动率与回撤是 PDF 数据字段支持的线性代理，不是基于协方差矩阵的组合波动率，也不是净值路径回撤。后续可在契约中把稳健性/绩效约束设为可选项或替换为更完整的风险模型。